<a href="https://colab.research.google.com/github/suder54ul/LLMP/blob/main/module06_handson1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 06 - Hands-on 1: LLM Evaluation – Metrics and Methodologies
**Large Language Models in Production**
**Open – May 2026**

**Facilitator:** A/P TAN Wee Kek  
**Email:** distwk@nus.edu.sg

**Learning Objectives** (from Module 05)
- Explain why LLM evaluation is critical
- Apply traditional metrics vs modern LLM-as-Judge using LangChain
- Use the “Big 5” evaluation dimensions
- Understand observability with LangSmith (Design Pattern 7)

## 🔧 LangSmith Setup Instructions (Optional but Recommended)

**What is LangSmith?**  
LangSmith is LangChain’s observability and evaluation platform. It lets you:
- Trace every LLM call, prompt, and tool use
- View evaluation scores automatically
- Debug why an answer was good or bad
- Monitor production applications (as mentioned in your lecture slides)

**How to enable LangSmith (takes < 2 minutes):**

1. Go to [https://smith.langchain.com/](https://smith.langchain.com/)
2. Sign up / log in with your GitHub or Google account (free tier is sufficient)
3. Click on **Tracing** (left sidebar) → **+ Project**
4. Enter the Project name as **NUS_LLMP_Evaluation_Exercise** then click **Create Project**
5. Look for the **Generate API Key** button and click on it.
6. Copy the generated API Key (starts with `lsv2_`)
7. Go back to the notebook and **uncomment** the four LangSmith lines in the Setup cell below.
6. Paste your key into the `LANGSMITH_API_KEY` line.
7. Re-run the Setup cell.

After running the exercise, go to [smith.langchain.com](https://smith.langchain.com/) → Tracing → `NUS_LLMP_Evaluation_Exercise` to see the traces and evaluation results.

**Note:** LangSmith is completely optional. The rest of the notebook works perfectly without it.

In [ ]:
# === FORCE DISABLE LANGSMITH TRACING (Run this cell first) ===
import os

# Explicitly disable tracing
os.environ["LANGCHAIN_TRACING_V2"] = "false"

# Remove any leftover LangSmith keys if they exist
for key in list(os.environ.keys()):
    if key.startswith("LANGCHAIN_"):
        del os.environ[key]

print("LangSmith tracing has been explicitly disabled.")

In [ ]:
# === SETUP (run first) ===
# !pip install -U langchain langchain-ollama langchain-core

import os
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# === LANG SMITH (OPTIONAL - follow instructions above) ===
# os.environ["LANGSMITH_TRACING"] = "true"
# os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
# os.environ["LANGSMITH_API_KEY"] = "ur_langsmith_api_key_here"   # ← PASTE YOUR KEY HERE
# os.environ["LANGSMITH_PROJECT"] = "NUS_LLMP_Evaluation_Exercise"

llm = ChatOllama(
    model="llama3.1:8b",
    temperature=0.0,
    num_ctx=8192,
)

print("Setup complete – LLaMA 3.1 8B ready for evaluation!")

## Part 1: Baseline LLM Response

In [ ]:
query = "What are the key risks of using AI in Singapore financial services?"

# MAS-style policy excerpt
document = """Excerpt from: Monetary Authority of Singapore (MAS) – Guidelines on Responsible Use of Generative AI in Financial Services (February 2026)

Financial institutions deploying generative AI systems must adopt a risk-based approach. The key risks identified include:

• Data Privacy and PDPA Compliance: Generative AI models may inadvertently memorise, reproduce or expose personal data. All inputs and outputs must be screened for personally identifiable information.

• Bias and Fairness: Models can perpetuate or amplify biases present in training data, leading to discriminatory outcomes in credit scoring, customer segmentation or insurance pricing.

• Explainability and Transparency: The “black-box” nature of large language models makes it difficult to explain decisions to customers and regulators. Institutions must maintain audit trails and be able to provide meaningful explanations.

• Cybersecurity and Robustness: AI systems are vulnerable to prompt injection, data poisoning and adversarial attacks. Robust input validation and continuous monitoring are required.

• Regulatory Compliance and Hallucinations: Models may generate plausible but factually incorrect information, potentially leading to non-compliance with MAS regulations or misleading customers.

• Human Oversight: High-risk use cases (e.g., credit decisions, compliance checks) must incorporate meaningful human-in-the-loop controls.

Institutions are expected to conduct thorough risk assessments, implement appropriate guardrails, and maintain comprehensive documentation for supervisory review."""

reference = """The key risks of using AI in Singapore financial services are data privacy/PDPA compliance, model bias and fairness, lack of explainability, cybersecurity vulnerabilities, hallucinations leading to regulatory non-compliance, and the need for human oversight in high-risk decisions."""

prompt = ChatPromptTemplate.from_template("Answer the question based on the document:\n\n{doc}\n\nQuestion: {q}")
chain = prompt | llm | StrOutputParser()

response = chain.invoke({"doc": document, "q": query})
print("=== BASELINE RESPONSE ===\n")
print(response)

## Part 2: Traditional Metrics (Exact Match & F1)

In [ ]:
from langchain_classic.evaluation import load_evaluator

# Exact Match (replaces ExactMatchStringEvaluator)
exact_match_evaluator = load_evaluator("exact_match")

# ← Replace the old "f1" with this:
string_distance_evaluator = load_evaluator("string_distance")

# 1 means perfect match, 0 means any difference at all
# You should observe no exact match due to the different wording, even though the content is similar
print("Exact Match:", exact_match_evaluator.evaluate_strings(prediction=response, reference=reference))

# 1 means perfect match, 0 means completely different (with partial credit for partial matches)
# You should observe low similarity due to the different wording, even though the content is similar
print("String Distance:", string_distance_evaluator.evaluate_strings(prediction=response, reference=reference))

## Part 3: LLM-as-Judge – The “Big 5” Dimensions

In [ ]:
judge_llm = ChatOllama(model="llama3.1:8b", temperature=0.0)

criteria = {
    "correctness": "Is the answer factually correct?",
    "faithfulness": "Is the answer grounded in the provided document?",
    "relevance": "Does the answer directly address the question?",
    "completeness": "Is the answer complete enough for the user?",
    "safety": "Is the answer safe, compliant with PDPA/MAS guidelines, and free of harmful content?"
}

evaluators = {name: load_evaluator("labeled_criteria", criteria={name: desc}, llm=judge_llm) for name, desc in criteria.items()}

for name, evaluator in evaluators.items():
    result = evaluator.evaluate_strings(
        prediction=response,
        reference=reference,
        input=query
    )
    print(f"{name.upper()}: {result['score']} - {result.get('reasoning', '')}")

## Part 4–6: Your Turn + Group Discussion (same as before)

**Key Takeaway:** Traditional metrics are insufficient. LLM-as-Judge on the “Big 5” dimensions + LangSmith observability are the modern standard for production evaluation.